In [53]:
"""
T5-Flan-small Authorship Classification Script

This script uses T5-Flan-small for direct authorship classification
using prompt-based approach instead of embeddings.
"""

import torch
import numpy as np
from transformers import T5ForConditionalGeneration, T5Tokenizer, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm
import warnings
import re
import random
from torch.utils.data import Dataset
warnings.filterwarnings('ignore')

class AuthorshipDataset(Dataset):
    """Dataset class for T5 fine-tuning"""

    def __init__(self, texts, authors, tokenizer, max_length=512):
        self.texts = texts
        self.authors = authors
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        author = self.authors[idx]

        # Create better input prompt with clear instruction
        input_text = f"Who wrote this text? Text: {text[:300]}"
        target_text = author

        # Tokenize input
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Tokenize target
        target_encoding = self.tokenizer(
            target_text,
            max_length=50,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': input_encoding['input_ids'].flatten(),
            'attention_mask': input_encoding['attention_mask'].flatten(),
            'labels': target_encoding['input_ids'].flatten()
        }

class T5AuthorshipClassifier:
    """T5-Flan-small wrapper for authorship classification with fine-tuning"""

    def __init__(self, model_name='google/flan-t5-small'):
        """
        Initialize T5-Flan-small model

        Args:
            model_name (str): Hugging Face model name
        """
        print(f"Loading T5-Flan-small model: {model_name}")
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        print(f"Model loaded on device: {self.device}")

    def fine_tune(self, texts, authors, num_epochs=5, batch_size=2):
        """
        Fine-tune T5 model on authorship classification

        Args:
            texts (list): List of training texts
            authors (list): List of corresponding authors
            num_epochs (int): Number of training epochs
            batch_size (int): Training batch size
        """
        print(f"Starting fine-tuning for {num_epochs} epochs...")
        print(f"Training on {len(texts)} examples with {len(set(authors))} authors")

        # Create dataset
        dataset = AuthorshipDataset(texts, authors, self.tokenizer)

        # Training arguments with better settings
        training_args = TrainingArguments(
            output_dir='./t5_authorship_model',
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=50,
            weight_decay=0.01,
            learning_rate=5e-5,  # Lower learning rate for better fine-tuning
            logging_dir='./logs',
            logging_steps=5,
            save_steps=100,
            eval_strategy="no",
            save_total_limit=2,
            dataloader_drop_last=False,
            remove_unused_columns=False,
        )

        # Create trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=dataset,
            tokenizer=self.tokenizer,
        )

        # Train the model
        trainer.train()

        # Set model to eval mode
        self.model.eval()
        print("Fine-tuning completed!")

        return trainer

    def classify_texts(self, texts, batch_size=8):
        """
        Classify texts using fine-tuned T5-Flan-small

        Args:
            texts (list): List of text strings
            batch_size (int): Batch size for processing

        Returns:
            list: Predicted authors for each text
        """
        predictions = []

        print(f"Classifying {len(texts)} texts using fine-tuned T5-Flan-small...")

        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]

            for text in batch_texts:
                # Create better prompt for authorship classification
                prompt = f"Who wrote this text? Text: {text[:300]}"

                # Tokenize prompt
                inputs = self.tokenizer(
                    prompt,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=512
                )
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                # Generate prediction with better parameters
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_length=30,
                        num_beams=3,
                        do_sample=False,
                        temperature=0.7,
                        pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id
                    )

                    # Decode prediction
                    prediction = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                    predictions.append(prediction.strip())

        return predictions

    def test_on_samples(self, test_texts, test_authors, num_samples=10):
        """
        Test the model on specific samples from test set

        Args:
            test_texts (list): List of test texts
            test_authors (list): List of test authors
            num_samples (int): Number of samples to test

        Returns:
            list: Results of testing
        """
        print(f"\nTesting on {num_samples} samples from test set...")

        # Take first num_samples from test set
        sample_texts = test_texts[:num_samples]
        sample_authors = test_authors[:num_samples]

        results = []
        for i, (text, true_author) in enumerate(zip(sample_texts, sample_authors)):
            # Get prediction
            prediction = self.classify_texts([text])[0]

            # Check if correct
            is_correct = prediction.lower().strip() == true_author.lower().strip()

            result = {
                'text': text[:100] + "..." if len(text) > 100 else text,
                'true_author': true_author,
                'predicted_author': prediction,
                'correct': is_correct
            }
            results.append(result)

            print(f"\nSample {i+1}:")
            print(f"Text: {result['text']}")
            print(f"True Author: {result['true_author']}")
            print(f"Predicted Author: {result['predicted_author']}")
            print(f"Correct: {result['correct']}")

        # Calculate accuracy on samples
        correct_count = sum(1 for r in results if r['correct'])
        accuracy = correct_count / len(results)
        print(f"\nSample Accuracy: {accuracy:.3f} ({correct_count}/{len(results)})")

        return results

class T5Evaluator:
    """Evaluator for T5 model performance"""

    def __init__(self, t5_model):
        self.t5_model = t5_model


    def evaluate_classification(self, texts, authors, test_size=0.2):
        """
        Evaluate T5 classification performance

        Args:
            texts (list): List of text strings
            authors (list): List of author names
            test_size (float): Test set proportion

        Returns:
            dict: Classification evaluation metrics
        """
        print("Evaluating T5 classification performance...")

        # Split data
        train_texts, test_texts, train_authors, test_authors = train_test_split(
            texts, authors, test_size=test_size, random_state=42
        )

        # Get predictions from T5
        y_pred = self.t5_model.classify_texts(test_texts)

        # Calculate accuracy
        correct = sum(1 for pred, true in zip(y_pred, test_authors)
                     if pred.lower().strip() == true.lower().strip())
        accuracy = correct / len(test_authors)

        return {
            'accuracy': accuracy,
            'y_true': test_authors,
            'y_pred': y_pred,
            'correct_predictions': correct,
            'total_predictions': len(test_authors)
        }


def load_stylometric_dataset(csv_path='stylometric_dataset.csv'):
    """
    Load the stylometric dataset from CSV file

    Args:
        csv_path (str): Path to the CSV file

    Returns:
        tuple: (texts, authors) - Lists of texts and corresponding authors
    """
    print(f"Loading stylometric dataset from {csv_path}...")

    # Read the CSV file
    df = pd.read_csv(csv_path)

    # Extract texts and authors
    texts = df['text'].tolist()
    authors = df['author'].tolist()

    # Convert author names to numeric IDs for easier processing
    unique_authors = list(set(authors))
    author_to_id = {author: idx for idx, author in enumerate(unique_authors)}
    author_ids = [author_to_id[author] for author in authors]

    print(f"Loaded {len(texts)} texts from {len(unique_authors)} authors")
    print(f"Authors: {unique_authors}")

    return texts, author_ids, unique_authors


def main():
    """Main testing function"""
    print("=" * 60)
    print("T5-Flan-small Authorship Classification with Fine-tuning")
    print("=" * 60)

    # Initialize T5 model
    t5_model = T5AuthorshipClassifier()

    # Load stylometric dataset
    print("\n1. Loading stylometric dataset...")
    texts, author_ids, unique_authors = load_stylometric_dataset()

    # Convert author IDs back to names for training
    author_names = [unique_authors[aid] for aid in author_ids]

    print(f"Dataset: {len(texts)} texts from {len(unique_authors)} authors")
    print(f"Authors: {unique_authors}")

    # Split data: 80% training, 20% test
    print("\n2. Splitting dataset (80% training, 20% test)...")
    train_texts, test_texts, train_authors, test_authors = train_test_split(
        texts, author_names, test_size=0.2, random_state=42
    )

    print(f"Training set: {len(train_texts)} texts")
    print(f"Test set: {len(test_texts)} texts")

    # Fine-tune the model on training set
    print("\n3. Fine-tuning T5 model on training set...")
    trainer = t5_model.fine_tune(train_texts, train_authors, num_epochs=5, batch_size=2)

    # Test on 10 samples from test set
    print("\n4. Testing on 10 samples from test set...")
    sample_results = t5_model.test_on_samples(test_texts, test_authors, num_samples=10)

    # Initialize evaluator
    evaluator = T5Evaluator(t5_model)

    # Evaluate on entire test set
    print("\n5. Evaluating on entire test set...")
    classification_results = evaluator.evaluate_classification(test_texts, test_authors)

    print(f"\nFull Test Set Results:")
    print(f"  Accuracy: {classification_results['accuracy']:.3f}")
    print(f"  Correct: {classification_results['correct_predictions']}/{classification_results['total_predictions']}")

    print("\n" + "=" * 60)
    print("Fine-tuning and testing completed successfully!")
    print("=" * 60)

if __name__ == "__main__":
    main()

T5-Flan-small Authorship Classification with Fine-tuning
Loading T5-Flan-small model: google/flan-t5-small
Model loaded on device: cuda

1. Loading stylometric dataset...
Loading stylometric dataset from stylometric_dataset.csv...
Loaded 400 texts from 10 authors
Authors: ['Dr. Margaret Chen', 'Dr. Robert Stevens', 'Mike Davis', 'Sarah Kim', 'Alex Thompson', 'Jake Martinez', 'James Wilson', 'Nina Rodriguez', 'Professor Elizabeth Hartwell', 'Lisa Chang']
Dataset: 400 texts from 10 authors
Authors: ['Dr. Margaret Chen', 'Dr. Robert Stevens', 'Mike Davis', 'Sarah Kim', 'Alex Thompson', 'Jake Martinez', 'James Wilson', 'Nina Rodriguez', 'Professor Elizabeth Hartwell', 'Lisa Chang']

2. Splitting dataset (80% training, 20% test)...
Training set: 320 texts
Test set: 80 texts

3. Fine-tuning T5 model on training set...
Starting fine-tuning for 5 epochs...
Training on 320 examples with 10 authors


Step,Training Loss
5,43.829900
10,43.230600
15,43.624000
20,40.024200
25,39.363800
30,37.147500
35,34.702000
40,33.881500
45,30.784700
50,27.484900


Fine-tuning completed!

4. Testing on 10 samples from test set...

Testing on 10 samples from test set...
Classifying 1 texts using fine-tuned T5-Flan-small...


100%|██████████| 1/1 [00:00<00:00,  2.35it/s]



Sample 1:
Text: Dr. Robert Stevens: This is a summary of the procedures for a hospital. Dr. Stevens explained a meth...
True Author: Dr. Robert Stevens
Predicted Author: Dr. Robert Stevens
Correct: True
Classifying 1 texts using fine-tuned T5-Flan-small...


...


Sample 9:
Text: Nina Rodriguez is discussing creative projects. I thought I would do a few sketches for a little gir...
True Author: Nina Rodriguez
Predicted Author: Nina Rodriguez
Correct: True
Classifying 1 texts using fine-tuned T5-Flan-small...


100%|██████████| 1/1 [00:00<00:00, 12.07it/s]



Sample 10:
Text: Alex Thompson: We're talking about the new products/services market, and why this is so important. H...
True Author: Alex Thompson
Predicted Author: Alex Thompson
Correct: True

Sample Accuracy: 0.600 (6/10)

5. Evaluating on entire test set...
Evaluating T5 classification performance...
Classifying 16 texts using fine-tuned T5-Flan-small...


100%|██████████| 2/2 [00:02<00:00,  1.43s/it]


Full Test Set Results:
  Accuracy: 0.688
  Correct: 11/16

Fine-tuning and testing completed successfully!


In [25]:
from google.colab import files

uploaded = files.upload()

Saving stylometric_dataset.csv to stylometric_dataset.csv


In [40]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [45]:
class STARModel:
    """STAR Model wrapper for easy testing and evaluation"""

    def __init__(self, model_name='AIDA-UPM/star'):
        """
        Initialize STAR model

        Args:
            model_name (str): Hugging Face model name
        """
        print(f"Loading STAR model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained('roberta-large')
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        print(f"Model loaded on device: {self.device}")

    def extract_embeddings(self, texts, batch_size=32):
        """
        Extract style embeddings from texts

        Args:
            texts (list): List of text strings
            batch_size (int): Batch size for processing

        Returns:
            np.ndarray: Style embeddings (n_texts, 1024)
        """
        embeddings = []

        print(f"Extracting embeddings for {len(texts)} texts...")
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]

            # Tokenize batch
            inputs = self.tokenizer(
                batch_texts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            # Extract embeddings
            with torch.no_grad():
                outputs = self.model(**inputs)
                batch_embeddings = outputs.pooler_output.cpu().numpy()
                embeddings.append(batch_embeddings)

        return np.vstack(embeddings)

In [44]:
class STAREvaluator:
    """Evaluator for STAR model performance"""

    def __init__(self, star_model):
        self.star_model = star_model

    def evaluate_classification(self, embeddings, labels, test_size=0.2):
        """
        Evaluate classification performance

        Args:
            embeddings (np.ndarray): Style embeddings
            labels (list): Author labels
            test_size (float): Test set proportion

        Returns:
            dict: Classification evaluation metrics
        """
        print("Evaluating classification performance...")

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            embeddings, labels, test_size=test_size, random_state=42, stratify=labels
        )

        # Train classifier
        classifier = XGBClassifier(random_state=42, eval_metric='mlogloss')
        classifier.fit(X_train, y_train)

        # Predict and evaluate
        y_pred = classifier.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        return {
            'accuracy': accuracy,
            'classification_report': classification_report(y_test, y_pred),
            'y_true': y_test,
            'y_pred': y_pred
        }

In [43]:
def load_stylometric_dataset(csv_path='stylometric_dataset.csv'):
    """
    Load the stylometric dataset from CSV file

    Args:
        csv_path (str): Path to the CSV file

    Returns:
        tuple: (texts, authors) - Lists of texts and corresponding authors
    """
    print(f"Loading stylometric dataset from {csv_path}...")

    # Read the CSV file
    df = pd.read_csv(csv_path)

    # Extract texts and authors
    texts = df['text'].tolist()
    authors = df['author'].tolist()

    # Convert author names to numeric IDs for easier processing
    unique_authors = list(set(authors))
    author_to_id = {author: idx for idx, author in enumerate(unique_authors)}
    author_ids = [author_to_id[author] for author in authors]

    print(f"Loaded {len(texts)} texts from {len(unique_authors)} authors")
    print(f"Authors: {unique_authors}")

    return texts, author_ids, unique_authors

In [46]:
def main():
    """Main testing function"""
    print("=" * 60)
    print("STAR (Style Transformer for Authorship Representations) Testing")
    print("=" * 60)

    # Initialize STAR model
    star_model = STARModel()

    # Load stylometric dataset
    print("\n1. Loading stylometric dataset...")
    texts, author_ids, unique_authors = load_stylometric_dataset()

    print(f"Dataset: {len(texts)} texts from {len(unique_authors)} authors")

    # Extract embeddings
    print("\n2. Extracting style embeddings...")
    embeddings = star_model.extract_embeddings(texts)

    print(f"Embeddings shape: {embeddings.shape}")

    # Initialize evaluator
    evaluator = STAREvaluator(star_model)

    # Evaluate classification performance
    print("\n3. Evaluating classification performance...")
    classification_results = evaluator.evaluate_classification(embeddings, author_ids)

    print(f"Classification Results:")
    print(f"  Accuracy: {classification_results['accuracy']:.3f}")

    # Print detailed classification report
    print("\n4. Detailed Classification Report:")
    print(classification_results['classification_report'])

    print("\n" + "=" * 60)
    print("Testing completed successfully!")
    print("=" * 60)

In [47]:
if __name__ == "__main__":
    main()

STAR (Style Transformer for Authorship Representations) Testing
Loading STAR model: AIDA-UPM/star
Model loaded on device: cuda

1. Loading stylometric dataset...
Loading stylometric dataset from stylometric_dataset.csv...
Loaded 400 texts from 10 authors
Authors: ['Dr. Margaret Chen', 'Dr. Robert Stevens', 'Mike Davis', 'Sarah Kim', 'Alex Thompson', 'Jake Martinez', 'James Wilson', 'Nina Rodriguez', 'Professor Elizabeth Hartwell', 'Lisa Chang']
Dataset: 400 texts from 10 authors

2. Extracting style embeddings...
Extracting embeddings for 400 texts...


100%|██████████| 13/13 [00:12<00:00,  1.08it/s]


Embeddings shape: (400, 1024)

3. Evaluating classification performance...
Evaluating classification performance...
Classification Results:
  Accuracy: 0.750

4. Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.75      0.71         8
           1       0.88      0.88      0.88         8
           2       0.83      0.62      0.71         8
           3       0.89      1.00      0.94         8
           4       0.56      0.62      0.59         8
           5       0.86      0.75      0.80         8
           6       0.40      0.50      0.44         8
           7       1.00      0.75      0.86         8
           8       0.86      0.75      0.80         8
           9       0.78      0.88      0.82         8

    accuracy                           0.75        80
   macro avg       0.77      0.75      0.75        80
weighted avg       0.77      0.75      0.75        80


Testing completed successfully!
